In [1]:
# Install dependencies
%pip install anthropic python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 4.4 MB/s eta 0:00:00


In [2]:
# Load env variables
from anthropic import Anthropic
from google.colab import userdata
from IPython.display import Markdown
from IPython.display import JSON
import json


In [3]:
# 1. Retrieve the key from Colab Secrets
my_anthropic_key   = userdata.get('ANTHROPIC_API_KEY')
my_anthropic_model = userdata.get('CLAUDE_MODEL')


In [4]:

# Create an API client
client = Anthropic(
    api_key=my_anthropic_key,
)

print("Anthropic client initialized successfully!")


Anthropic client initialized successfully!


In [5]:
# define a system prompt
#system_prompt = """
#You are a creative screenwriter for feature films.
#"""
system_prompt = None

In [6]:
# Utility functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, stop_sequences=None):
    params = {
        "model": my_anthropic_model,
        "max_tokens": 1000,
        "messages": messages
    }

    if system:
        params["system"] = system

    # Add stop_sequences if provided
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return message.content[0].text

In [7]:
# Generate dataset
def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex",
        "solution_criteria": "Key criteria for evaluating the solution"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [8]:
# Testing dataset
dataset = generate_dataset()
print(dataset)

with open("dataset.json", "w") as f:
    json.dump(dataset, f)

[{'task': 'Extract all S3 bucket names from AWS CloudTrail logs in JSON format', 'format': 'regex', 'solution_criteria': 'The regex should accurately match S3 bucket ARNs (arn:aws:s3:::bucket-name) and extract just the bucket name, handling hyphens and numbers in bucket names'}, {'task': 'Create a CloudFormation template snippet for launching a basic EC2 instance with a specified AMI ID, instance type, and security group', 'format': 'json', 'solution_criteria': 'The JSON should be valid CloudFormation syntax, include required EC2 properties (ImageId, InstanceType, SecurityGroupIds), and be deployable without errors'}, {'task': "Write a Python function that parses an AWS IAM policy document and returns a list of all resource ARNs that have the 's3:*' action granted", 'format': 'python', 'solution_criteria': "The function should correctly parse nested IAM policy structures, identify statements with 's3:*' actions, extract resource ARNs, and handle wildcards and multiple resources"}]


In [9]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Criteria you should use to evaluate the solution:
<criteria>
{test_case["solution_criteria"]}
</criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [10]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # TODO - Grading
    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
        "format": test_case["format"]
    }

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    return results

In [11]:
def run_dataset_evaluation():

  with open("dataset.json", "r") as f:
    dataset = json.load(f)

  results = run_eval(dataset)

  return results

In [12]:
from textwrap import indent

results = run_dataset_evaluation()

print(json.dumps(results, indent=2))


[
  {
    "output": "\nimport json\nimport re\nimport sys\n\ndef extract_s3_bucket_names(cloudtrail_logs):\n    \"\"\"Extract S3 bucket names from CloudTrail logs.\"\"\"\n    bucket_names = set()\n    \n    # Parse JSON if it's a string\n    if isinstance(cloudtrail_logs, str):\n        try:\n            logs = json.loads(cloudtrail_logs)\n        except json.JSONDecodeError:\n            return list(bucket_names)\n    else:\n        logs = cloudtrail_logs\n    \n    # Handle both single event and Records array\n    events = logs.get('Records', [logs]) if isinstance(logs, dict) else logs\n    \n    # Regex patterns for S3 bucket names in various CloudTrail fields\n    patterns = [\n        r'\"bucket\":\\s*\"([a-z0-9.\\-]+)\"',\n        r'\"bucketName\":\\s*\"([a-z0-9.\\-]+)\"',\n        r'\"Bucket\":\\s*\"([a-z0-9.\\-]+)\"',\n        r'\"name\":\\s*\"([a-z0-9.\\-]+)\"(?=.*bucket)',\n        r'arn:aws:s3:::([a-z0-9.\\-]+)(?:[/\"]|$)',\n        r'\"requestParameters\":\\s*\\{[^}]*\"buck

In [13]:
# Functions to validate the output structure
import re
import ast

def clean_llm_output(text):
    """Strips markdown code blocks from the LLM output."""
    text = text.strip()

    # Remove the starting ```, ```python, or ```json (case-insensitive)
    text = re.sub(r'^```(?:python|json)?\s*', '', text, flags=re.IGNORECASE)

    # Remove the trailing ```
    text = re.sub(r'\s*```$', '', text)

    return text.strip()


def validate_json(text):
    try:
        json.loads(clean_llm_output(text))
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(clean_llm_output(text))
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(clean_llm_output(text))
        return 10
    except re.error:
        return 0


def grade_syntax(response):
    format = response["format"]
    if format == "json":
        return validate_json(response["output"])
    elif format == "python":
        return validate_python(response["output"])
    else:
        return validate_regex(response["output"])


In [14]:
from statistics import mean

average_model_score = mean([result["score"] for result in results])
print(f"Average score: {average_model_score}")


Average score: 6.333333333333333


In [15]:

syntax_score_sum = 0
num_scores = 0
for result in results:
        syntax_score = grade_syntax(result)
        print(f"Syntax score: {syntax_score} for the response format {result["format"]}")
        syntax_score_sum = syntax_score_sum + syntax_score
        num_scores = num_scores + 1

average_syntax_score = syntax_score_sum / num_scores

score = (average_model_score + average_syntax_score) / 2
print(f"Model score: {average_model_score}")
print(f"Syntax score: {average_syntax_score}")
print(f"Final score: {score}")


Syntax score: 0 for the response format regex
Syntax score: 10 for the response format json
Syntax score: 10 for the response format python
Model score: 6.333333333333333
Syntax score: 6.666666666666667
Final score: 6.5
